# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

## Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import warnings
warnings.filterwarnings('ignore')  # (For some pandas warnings)

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata and Croissant definition
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"Version: {getattr(metadata, 'version', 'Unknown')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'Unknown')}")


## 2. Data Overview

Review available record sets, their `@id`s, and the fields/columns available in each. All entities are referenced by their `@id`, following the Croissant schema.

In [ ]:
# List record sets, their fields, and columns by @id

print("Record sets in dataset:")
record_sets = [r for r in metadata.record_sets]  # List of mlc.RecordSet objects
for idx, rs in enumerate(record_sets):
    print(f"{idx+1}. @id: {rs.id}  | Name: {getattr(rs, 'name', None)}")

    # List columns for this record set
    # Each column is an mlc.Column object
    if hasattr(rs, 'columns'):
        print("    Columns (@id):")
        for col in rs.columns:
            print(f"      - {col.id} (name: {getattr(col, 'name', col.id)})  [dataType: {getattr(col, 'data_type', 'unknown')}]")
    print()
# Save the @id of the main survey record set for further analysis
main_record_set_id = None
if record_sets:
    main_record_set_id = record_sets[0].id  # Select the first (main) record set
print(f"Selected main record set @id for analysis: {main_record_set_id}\n")

## 3. Data Extraction

Load data from each record set into Pandas DataFrames for analysis. Use the record set and column `@id`s from above.

In [ ]:
# List all record set @ids for extraction
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  (no records available)")
    print()
# The most likely main record set's DataFrame:
main_rs_df = dataframes.get(main_record_set_id)
if main_rs_df is not None:
    print(f"Columns in main DataFrame ({main_record_set_id}):\n{main_rs_df.columns.tolist()}")
    display(main_rs_df.head())

## 4. Exploratory Data Analysis (EDA)

Example: filter, normalize, group and summarise fields using the survey dataset. Please adapt the field `@id` and group field to suit the specific data (see column names printed above).

In [ ]:
# Set your numeric field @id and grouping field @id below
# You may need to adjust them (see column listing output above)

# Example using common mental health indices in these surveys, e.g.: 'gad7_total_score', 'phq9_total_score'
numeric_field = None
possible_numeric_names = [col for col in main_rs_df.columns if any(x in col.lower() for x in ['gad', 'phq', 'psq', 'score'])]
if possible_numeric_names:
    numeric_field = possible_numeric_names[0]

print(f"Using numeric field for EDA: {numeric_field}")

# Choose a grouping field (e.g., 'gender', 'age', or similar, depending on what's available)
possible_groups = [c for c in main_rs_df.columns if any(x in c.lower() for x in ['gender', 'sex', 'village', 'site', 'group', 'education'])]
group_field = possible_groups[0] if possible_groups else None
print(f"Grouping by: {group_field}")

# Filtering configuration
threshold = 10

if numeric_field and numeric_field in main_rs_df.columns:
    filtered_df = main_rs_df[main_rs_df[numeric_field].astype(float) > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}: (showing first 5)")
    display(filtered_df.head(5))
    
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA. Please inspect 'main_rs_df.columns'.")

## 5. Visualization

Visualize numeric field distributions and groupings. Adapt the field `@id` to your dataset as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
# Plot distribution of the main numeric field
if numeric_field and numeric_field in main_rs_df.columns:
    plt.figure(figsize=(8, 5))
    main_rs_df[numeric_field].astype(float).hist(bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field, if available
    if group_field and group_field in main_rs_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=main_rs_df[group_field], y=main_rs_df[numeric_field].astype(float))
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable fields for charting. Please check column listing above.")

## 6. Conclusion

* You have loaded and explored the FAIR² Mental Health Survey Dataset using the `mlcroissant` Python library.
* You've listed record set and column `@id`s and explored the main survey data.
* Basic exploratory analysis and visualization were performed on the main numeric fields, such as GAD-7 or PHQ-9 scores if present.

You can further extend this notebook with more detailed analysis and modeling using the data and metadata described in the Croissant schema.